# 01 — Structured MIMIC-IV EDA

Interactive companion to `python -m src.eda.structured`. The script produces the citable, aggregate-only summary (`results/eda/structured_summary.json` + figures); this notebook is for drilling into it and asking follow-up questions.

> **Privacy rules (PhysioNet DUA) — read before running anything**
> 1. Never commit this notebook with outputs. Before `git add`: *Kernel → Restart & Clear Output*
>    (`tests/test_eda_privacy.py` fails if outputs are stored).
> 2. Don't print row-level data you'd paste anywhere (`df.head()`, note text, IDs). It's fine to look at it locally; it's not fine to copy it into a doc, chat, LLM prompt or slide.
> 3. Anything that leaves this notebook goes through `src.eda.privacy` (`safe_count`, `safe_rate`, `safe_quantiles`, …): counts < 11 are suppressed, no raw min/max.
> 4. Row-level caches live in `data/interim/eda_cache/` (gitignored). Never move them into `results/`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.config import load_config
from src.data.features import feature_columns
from src.eda import loaders as L, structured as S
from src.eda.privacy import *  # safe_* helpers

pd.set_option('display.max_columns', 60)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
summary = json.loads((ROOT / 'results/eda/structured_summary.json').read_text())
summary.keys()

## 1. The headline numbers (from the saved summary)

In [ ]:
pd.DataFrame(summary['cohort_flow']['steps'])

In [ ]:
o = summary['outcome']
{k: o[k] for k in ['all_cause_30d', 'unplanned_30d (model target)', 'planned_returns_removed', 'competing_risk_death']}

In [ ]:
pd.DataFrame(summary['missingness']['features_with_missing']).head(20)

In [ ]:
pd.DataFrame(summary['univariate_auroc']).head(20)

## 2. Load the row-level cohort (local only — cached, ~1 min)

`m` = the exact Stage 1 feature matrix (built from raw data with the pipeline's own code) plus descriptive columns (`race_group`, `insurance`, `discharge_location`, `year_group`, `had_icu_stay`, …).

In [ ]:
cfg = load_config()
matrix = L.load_feature_matrix_fresh(cfg)
m = S.enrich(matrix, L.load_admissions(cfg), L.load_patients(cfg), L.load_icustays(cfg))
feats = feature_columns(matrix)
TARGET = S.TARGET
m.shape

## 3. Your exploration

Pattern: compute on `m`, report through a `safe_*` helper. Examples:

In [ ]:
# Readmission rate by any categorical column, small groups folded into 'Other'
pd.DataFrame(rate_by_group(m, 'discharge_location', TARGET))

In [ ]:
# Two-way breakdown, e.g. age band x discharged to facility
m['age_x_facility'] = m['age_band'].astype(str) + ' / facility=' + m['discharged_to_facility'].astype(str)
pd.DataFrame(rate_by_group(m, 'age_x_facility', TARGET))

In [ ]:
# Distribution of a feature by outcome
pd.DataFrame({'readmitted': safe_quantiles(m.loc[m[TARGET]==1, 'los_days']),
              'not': safe_quantiles(m.loc[m[TARGET]==0, 'los_days'])})

### Open questions worth chasing (see session 37)
- Does the vitals missingness (non-ICU admissions) act as an ICU proxy in Stage 1's SHAP ranking?
- Is the 70+ discrimination gap a base-rate issue or a feature-coverage issue (missingness by age band)?
- How many 0–1-day 'readmissions' are actually transfers / administrative re-registrations?
- Competing risk: should post-discharge deaths be excluded, or reported as a sensitivity analysis?